In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, roc_curve, roc_auc_score, ConfusionMatrixDisplay)
import shap
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully.')

In [ ]:
# Load dataset — download from UCI if not present
# https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
# Place bank-additional-full.csv in the same directory, OR run:
#   !wget https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
#   !unzip bank-additional.zip

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip'
# Uncomment below if running for the first time:
# import urllib.request, zipfile, io
# data = urllib.request.urlopen(URL).read()
# z = zipfile.ZipFile(io.BytesIO(data))
# z.extractall('.')

df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')
print(f'Shape: {df.shape}')
df.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
print('Data types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nTarget distribution:\n', df['y'].value_counts())

In [ ]:
# Replace 'unknown' with NaN and impute with mode for categoricals
df.replace('unknown', np.nan, inplace=True)
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('y')  # keep target separate

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Encode target
df['y'] = (df['y'] == 'yes').astype(int)

# Label-encode all remaining categoricals
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('Encoding complete. Class balance:')
print(df['y'].value_counts(normalize=True).rename({0:'No',1:'Yes'}))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age distribution by subscription
df.groupby('y')['age'].plot(kind='kde', ax=axes[0], legend=True)
axes[0].set_title('Age Distribution by Subscription')
axes[0].set_xlabel('Age')
axes[0].legend(['No', 'Yes'])

# Campaign contacts distribution
df['campaign'].hist(bins=20, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Number of Contacts During Campaign')
axes[1].set_xlabel('campaign')

# Correlation heatmap (numeric cols)
num_cols = df.select_dtypes(include=np.number).columns[:8]
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', ax=axes[2], cmap='coolwarm')
axes[2].set_title('Correlation Heatmap (first 8 features)')

plt.tight_layout()
plt.savefig('task1_eda.png', dpi=120)
plt.show()

## 4. Model Building & Evaluation

In [ ]:
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Logistic Regression ───────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['No','Yes']))
print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['No','Yes']))

In [ ]:
# ── Confusion matrices & ROC curves ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, model, preds, name in zip(
    axes[0],
    [lr, rf],
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression', 'Random Forest']
):
    ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=ax,
                                            display_labels=['No','Yes'],
                                            colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}')

ax_roc = axes[1][0]
for model, Xeval, name, color in [
    (lr, X_test_sc, 'Logistic Regression', 'royalblue'),
    (rf, X_test,   'Random Forest',        'darkorange')
]:
    probs = model.predict_proba(Xeval)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax_roc.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color)

ax_roc.plot([0,1],[0,1],'k--', lw=0.8)
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curve Comparison')
ax_roc.legend()

# Feature importances
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).nlargest(10)
feat_imp.sort_values().plot(kind='barh', ax=axes[1][1], color='teal')
axes[1][1].set_title('Top 10 Feature Importances (Random Forest)')

plt.tight_layout()
plt.savefig('task1_evaluation.png', dpi=120)
plt.show()

## 5. Explainable AI — SHAP
Using SHAP TreeExplainer to explain 5 individual Random Forest predictions.

In [ ]:
# ── SHAP global summary ───────────────────────────────────────────────────────
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[1], X_test, show=False, plot_size=(10,6))
plt.title('SHAP Summary Plot (class=Yes)')
plt.tight_layout()
plt.savefig('task1_shap_summary.png', dpi=120)
plt.show()

In [ ]:
# ── SHAP waterfall for 5 individual predictions ───────────────────────────────
sample_indices = X_test.index[:5]
explanation = explainer(X_test.loc[sample_indices])

for i in range(5):
    plt.figure()
    shap.waterfall_plot(explanation[i], max_display=10, show=False)
    plt.title(f'SHAP Waterfall — Sample {i+1} (Actual={y_test.loc[sample_indices[i]]}, '
              f'Predicted={y_pred_rf[list(X_test.index).index(sample_indices[i])]})')
    plt.tight_layout()
    plt.savefig(f'task1_shap_sample_{i+1}.png', dpi=100)
    plt.show()
    print(f'Sample {i+1} explained.')